In [1]:
import pandas as pd
df_train = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/FewRel/Train/MINT/FewRel_full_binary_question_result.csv')
print(df_train.shape)
df_train.head(2)

(4000, 1829)


,id,relation,relation_id,text,subject,object,subj_start,subj_end,obj_start,obj_end,...,associated_with_region,artist_origin_city,reorganized_after_loss_of_papal_states,held_diplomatic_position_in_location,partnership_location,diplomatic_location_request,affiliated_with_location,participant_at_event_location,artist_origin_location,inventor_location_use
0,train_P1877_557,after a work by,P1877,"After reading Alan Glynn 's "" The Dark Fields ...",limitless,alan glynn,14,14,2,3,...,0,0,0,0,0,0,0,0,0,0
1,train_P1877_158,after a work by,P1877,"It was a sequel to the 1952 film "" Heidi "" , w...",heidi,johanna spyri,9,9,25,26,...,0,0,0,0,0,0,0,0,0,0


In [2]:
df_train['multi_labels']

0       ['inspired_by_author', 'adapted_from_author', ...
1       ['inspired_by_author', 'adapted_from_author', ...
2       ['inspired_by_author', 'adapted_from_author', ...
3       ['inspired_by_author', 'adapted_from_author', ...
4       ['inspired_by_author', 'adapted_from_author', ...
                              ...                        
3995    ['event_location_governance', 'government_juri...
3996    ['located_in', 'affiliated_with', 'administrat...
3997    ['occurred_in', 'adaptation_premiere_location'...
3998    ['institution_location', 'affiliated_with', 'p...
3999    ['entity_type_in_country', 'office_location', ...
Name: multi_labels, Length: 4000, dtype: object

In [3]:
import ast
import numpy as np

from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import pairwise_distances



def parse_multi_labels(x):
    """
    df_train['multi_labels']가 문자열 형태의 list인 경우 실제 list로 변환.
    예: "['Partnership', 'abbreviation']" -> ['Partnership', 'abbreviation']
    """
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, str):
        x = x.strip()

        if x == "" or x.lower() == "nan":
            return []

        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return [str(v).strip() for v in parsed if str(v).strip() != ""]
            return [str(parsed).strip()]
        except Exception:
            return [v.strip() for v in x.split(",") if v.strip() != ""]

    return []


df_train["multi_labels_parsed"] = df_train["multi_labels"].apply(parse_multi_labels)

print("example parsed labels:")
print(df_train["multi_labels_parsed"].head())


phrase_list = sorted(
    list({
        phrase
        for labels in df_train["multi_labels_parsed"]
        for phrase in labels
    })
)

phrase_to_idx = {p: i for i, p in enumerate(phrase_list)}
idx_to_phrase = {i: p for p, i in phrase_to_idx.items()}

n_instances = len(df_train)
n_phrases = len(phrase_list)

print("num instances:", n_instances)
print("num unique phrases:", n_phrases)

if n_phrases == 0:
    raise ValueError("multi_labels에서 phrase를 하나도 찾지 못했습니다.")


X_instance_phrase = np.zeros((n_instances, n_phrases), dtype=np.int8)

for i, labels in enumerate(df_train["multi_labels_parsed"]):
    for phrase in labels:
        if phrase in phrase_to_idx:
            j = phrase_to_idx[phrase]
            X_instance_phrase[i, j] = 1

print("X_instance_phrase shape:", X_instance_phrase.shape)


X_phrase_instance = X_instance_phrase.T

print("X_phrase_instance shape:", X_phrase_instance.shape)


jaccard_distance = pairwise_distances(
    X_phrase_instance.astype(bool),
    metric="jaccard"
)

jaccard_distance = np.nan_to_num(jaccard_distance, nan=1.0)
jaccard_similarity = 1.0 - jaccard_distance

print("jaccard_distance shape:", jaccard_distance.shape)
print("jaccard_similarity min/max:", jaccard_similarity.min(), jaccard_similarity.max())


def estimate_k_by_zhang2021(features, k0=300, random_state=42):
    """
    Zhang et al. (2021a) 방식의 unknown K 추정.

    features: clustering 대상 vector, shape = (num_items, dim)
              여기서는 phrase-instance binary vector를 사용.
    k0      : over-clustering에 사용할 큰 cluster 개수.
    """
    n = len(features)

    if n < 2:
        cluster_info = pd.DataFrame({
            "cluster_id": [0],
            "size": [n],
            "threshold": [n],
            "is_valid": [True]
        })
        return 1, np.zeros(n, dtype=int), cluster_info

    k0 = min(k0, n)

    kmeans = KMeans(
        n_clusters=k0,
        random_state=random_state,
        n_init="auto"
    )

    labels_k0 = kmeans.fit_predict(features)

    cluster_sizes = pd.Series(labels_k0).value_counts().sort_index()
    threshold = n / k0

    valid_clusters = cluster_sizes[cluster_sizes >= threshold].index.tolist()
    estimated_k = len(valid_clusters)
    estimated_k = max(1, estimated_k)

    cluster_info = pd.DataFrame({
        "cluster_id": cluster_sizes.index,
        "size": cluster_sizes.values,
        "threshold": threshold,
        "is_valid": cluster_sizes.values >= threshold
    })

    return estimated_k, labels_k0, cluster_info


features_for_k_estimation = normalize(
    X_phrase_instance.astype(float),
    norm="l2",
    axis=1
)

k0 = 300

estimated_k, labels_k0, cluster_info = estimate_k_by_zhang2021(
    features=features_for_k_estimation,
    k0=k0,
    random_state=42
)

print("Estimated K:", estimated_k)
print(cluster_info.head())
print(cluster_info["is_valid"].value_counts())


def run_agglomerative_precomputed(distance_matrix, n_clusters):
    """
    sklearn 버전에 따라 metric 또는 affinity 인자를 사용.
    """
    try:
        model = AgglomerativeClustering(
            n_clusters=n_clusters,
            metric="precomputed",
            linkage="average"
        )
    except TypeError:
        model = AgglomerativeClustering(
            n_clusters=n_clusters,
            affinity="precomputed",
            linkage="average"
        )

    return model.fit_predict(distance_matrix)


phrase_cluster_labels = run_agglomerative_precomputed(
    distance_matrix=jaccard_distance,
    n_clusters=estimated_k
)

phrase_cluster_df = pd.DataFrame({
    "phrase": phrase_list,
    "phrase_idx": np.arange(n_phrases),
    "phrase_cluster": phrase_cluster_labels,
    "support": X_phrase_instance.sum(axis=1)
})

print("phrase_cluster_df shape:", phrase_cluster_df.shape)
print(phrase_cluster_df.head())


representative_rows = (
    phrase_cluster_df
    .sort_values(["phrase_cluster", "support", "phrase"], ascending=[True, False, True])
    .groupby("phrase_cluster", as_index=False)
    .first()
)

representative_rows = representative_rows.rename(columns={
    "phrase": "representative_phrase",
    "support": "representative_support"
})

final_relation_list = representative_rows["representative_phrase"].tolist()

print("num final relations:", len(final_relation_list))
print(final_relation_list[:20])


cluster_to_rep_phrase = dict(
    zip(
        representative_rows["phrase_cluster"],
        representative_rows["representative_phrase"]
    )
)

phrase_to_cluster = dict(
    zip(phrase_cluster_df["phrase"], phrase_cluster_df["phrase_cluster"])
)


def assign_instance_to_phrase_cluster(labels):
    """
    instance의 multi_labels가 어떤 phrase cluster에 가장 많이 속하는지 계산.
    가장 score가 높은 cluster를 최종 cluster로 할당.
    """
    if not labels:
        return -1, None, 0

    cluster_counts = {}

    for phrase in labels:
        if phrase not in phrase_to_cluster:
            continue

        c = phrase_to_cluster[phrase]
        cluster_counts[c] = cluster_counts.get(c, 0) + 1

    if len(cluster_counts) == 0:
        return -1, None, 0

    best_cluster = sorted(
        cluster_counts.items(),
        key=lambda x: (-x[1], x[0])
    )[0][0]

    best_score = cluster_counts[best_cluster]
    best_phrase = cluster_to_rep_phrase[best_cluster]

    return best_cluster, best_phrase, best_score


assigned = df_train["multi_labels_parsed"].apply(assign_instance_to_phrase_cluster)

df_train["llm_multilabel_cluster"] = assigned.apply(lambda x: x[0])
df_train["llm_multilabel_relation"] = assigned.apply(lambda x: x[1])
df_train["llm_multilabel_score"] = assigned.apply(lambda x: x[2])

print(df_train[[
    "id",
    "relation",
    "multi_labels_parsed",
    "llm_multilabel_cluster",
    "llm_multilabel_relation",
    "llm_multilabel_score"
]].head())



example parsed labels:
0    [inspired_by_author, adapted_from_author, base...
1    [inspired_by_author, adapted_from_author, base...
2    [inspired_by_author, adapted_from_author, base...
3    [inspired_by_author, adapted_from_author, base...
4    [inspired_by_author, adapted_from_author, base...
Name: multi_labels_parsed, dtype: object
num instances: 4000
num unique phrases: 1808
X_instance_phrase shape: (4000, 1808)
X_phrase_instance shape: (1808, 4000)
jaccard_distance shape: (1808, 1808)
jaccard_similarity min/max: 0.0 1.0
Estimated K: 102
   cluster_id  size  threshold  is_valid
0           0     2   6.026667     False
1           1     5   6.026667     False
2           2     4   6.026667     False
3           3     3   6.026667     False
4           4     5   6.026667     False
is_valid
False    198
True     102
Name: count, dtype: int64
phrase_cluster_df shape: (1808, 4)
                    phrase  phrase_idx  phrase_cluster  support
0          abbreviation_of           0      

# Train Result

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_rand_score,
    v_measure_score,
    homogeneity_score,
    completeness_score,
    confusion_matrix
)
from scipy.optimize import linear_sum_assignment


def evaluate_clustering(df: pd.DataFrame, pred_col: str, gold_col: str = "relation"):

    
    non_na_df = df[~df[pred_col].isna()].copy()

    if len(non_na_df) == 0:
        raise ValueError(f"{pred_col}에 유효한 값이 없습니다.")

    y_true = non_na_df[gold_col].astype(str).values
    y_pred = non_na_df[pred_col].astype(str).values

    
    overall_purity = (
        non_na_df
        .groupby(pred_col)[gold_col]
        .value_counts()
        .groupby(level=0)
        .apply(lambda x: x.max())
        .sum()
        / len(non_na_df)
    )

    
    ari = adjusted_rand_score(y_true, y_pred)

    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v_f1 = v_measure_score(y_true, y_pred)

    
    def b3_precision_recall_f1(y_true, y_pred):

        true_groups = {}
        pred_groups = {}

        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            true_groups.setdefault(t, set()).add(i)
            pred_groups.setdefault(p, set()).add(i)

        P_sum, R_sum = 0.0, 0.0
        n = len(y_true)

        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            T = true_groups[t]
            C = pred_groups[p]
            inter = len(T & C)

            P_sum += inter / len(C) if len(C) > 0 else 0.0
            R_sum += inter / len(T) if len(T) > 0 else 0.0

        P = P_sum / n
        R = R_sum / n
        F1 = (2 * P * R / (P + R)) if (P + R) > 0 else 0.0

        return P, R, F1

    b3_p, b3_r, b3_f1 = b3_precision_recall_f1(y_true, y_pred)

    
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    hungarian_acc = cm[row_ind, col_ind].sum() / cm.sum()

    
    print(f"Purity : {overall_purity:.4f}")
    print(f"ACC    : {hungarian_acc:.4f}")
    
    
    print(f"B3-F1  : {b3_f1:.4f}")
    
    
    print(f"V-F1   : {v_f1:.4f}")
    print(f"ARI    : {ari:.4f}")

    return {
        "Purity": overall_purity,
        "ARI": ari,
        "V-hom": hom,
        "V-com": com,
        "V-F1": v_f1,
        "B3-P": b3_p,
        "B3-R": b3_r,
        "B3-F1": b3_f1,
        "ACC": hungarian_acc,
    }

In [5]:
df_eval = df_train

df_eval.loc[df_eval["llm_multilabel_cluster"] == -1, "llm_multilabel_cluster"] = np.nan

metrics_cluster = evaluate_clustering(
    df=df_eval,
    pred_col="llm_multilabel_cluster",
    gold_col="relation"
)

metrics_cluster

Purity : 0.0447
ACC    : 0.0445
B3-F1  : 0.0669
V-F1   : 0.1365
ARI    : 0.0037


{'Purity': 0.04473085670962851,
 'ARI': 0.00365502578777531,
 'V-hom': 0.07679793906692774,
 'V-com': 0.6120441048916263,
 'V-F1': 0.13647171012856335,
 'B3-P': 0.03480209371000846,
 'B3-R': 0.8763715352640761,
 'B3-F1': 0.06694566946452271,
 'ACC': 0.04447814000505433}

# Test set evaluation

In [12]:
test_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/FewRel/Test/FewRel_test_detail_phrase.csv')
print(test_df.shape)
test_df.head(2)

(1600, 14)


,id,relation,relation_id,text,subject,object,subj_start,subj_end,obj_start,obj_end,subj_id,obj_id,token,gpt4_phrase
0,train_P1877_500,after a work by,P1877,"In 1998 , she starred opposite Elisabeth Shue ...",cousin bette,balzac,16,17,13,13,Q835676,Q9711,"['In', '1998', ',', 'she', 'starred', 'opposit...",literary work authored by
1,train_P1877_396,after a work by,P1877,"He directed Disney 's "" Tom and Huck "" in 1995...",tom and huck,mark twain,5,7,15,16,Q1616185,Q7245,"['He', 'directed', 'Disney', ""'s"", '""', 'Tom',...","adaptation source of the film ""Tom and Huck"" b..."


In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

if "gpt4_phrase" not in test_df.columns:
    raise ValueError("test_df에 'gpt4_phrase' 컬럼이 없습니다.")

if "relation" not in test_df.columns:
    raise ValueError("test_df에 gold label 컬럼 'relation'이 없습니다.")

if "phrase" not in phrase_cluster_df.columns:
    raise ValueError("phrase_cluster_df에 'phrase' 컬럼이 없습니다.")

if "phrase_cluster" not in phrase_cluster_df.columns:
    raise ValueError("phrase_cluster_df에 'phrase_cluster' 컬럼이 없습니다.")

if "representative_phrase" not in representative_rows.columns:
    raise ValueError("representative_rows에 'representative_phrase' 컬럼이 없습니다.")

train_phrases = phrase_cluster_df["phrase"].fillna("").astype(str).tolist()
test_phrases = test_df["gpt4_phrase"].fillna("").astype(str).tolist()

print("num train phrases:", len(train_phrases))
print("num test instances:", len(test_phrases))

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

train_embeddings = model.encode(
    train_phrases,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

test_embeddings = model.encode(
    test_phrases,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("train_embeddings shape:", train_embeddings.shape)
print("test_embeddings shape:", test_embeddings.shape)

similarity_matrix = test_embeddings @ train_embeddings.T

nearest_train_phrase_idx = similarity_matrix.argmax(axis=1)
nearest_train_phrase_similarity = similarity_matrix.max(axis=1)

test_df["nearest_train_phrase_idx"] = nearest_train_phrase_idx
test_df["nearest_train_phrase"] = [
    train_phrases[idx] for idx in nearest_train_phrase_idx
]
test_df["nearest_train_phrase_similarity"] = nearest_train_phrase_similarity

phrase_to_cluster = dict(
    zip(
        phrase_cluster_df["phrase"].astype(str),
        phrase_cluster_df["phrase_cluster"]
    )
)

cluster_to_rep_phrase = dict(
    zip(
        representative_rows["phrase_cluster"],
        representative_rows["representative_phrase"]
    )
)

test_df["llm_multilabel_cluster"] = test_df["nearest_train_phrase"].map(phrase_to_cluster)
test_df["llm_multilabel_relation"] = test_df["llm_multilabel_cluster"].map(cluster_to_rep_phrase)

print(test_df[[
    "relation",
    "gpt4_phrase",
    "nearest_train_phrase",
    "nearest_train_phrase_similarity",
    "llm_multilabel_cluster",
    "llm_multilabel_relation"
]].head(10))

num train phrases: 1808
num test instances: 1600


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

train_embeddings shape: (1808, 384)
test_embeddings shape: (1600, 384)
          relation                                        gpt4_phrase  \
0  after a work by                          literary work authored by   
1  after a work by  adaptation source of the film "Tom and Huck" b...   
2  after a work by                          literary work authored by   
3  after a work by                  adapted from the literary work of   
4  after a work by  original 1974 film adaptation is based on the ...   
5  after a work by                    original fairy tale authored by   
6  after a work by                          original work authored by   
7  after a work by                   adaptation of a novel written by   
8  after a work by             movie "Dagon" is inspired by Lovecraft   
9  after a work by                     is based on a novel written by   

            nearest_train_phrase  nearest_train_phrase_similarity  \
0         work_written_by_author                        

In [14]:
metrics_cluster = evaluate_clustering(
    df=test_df,
    pred_col="llm_multilabel_cluster",
    gold_col="relation"
)

metrics_cluster

Purity : 0.2794
ACC    : 0.2687
B3-F1  : 0.2891
V-F1   : 0.5380
ARI    : 0.0804


{'Purity': 0.279375,
 'ARI': 0.08044643356867431,
 'V-hom': 0.4561361089037682,
 'V-com': 0.6555435700943676,
 'V-F1': 0.537955490108821,
 'B3-P': 0.2025949860800878,
 'B3-R': 0.5047500000000004,
 'B3-F1': 0.28913704411936314,
 'ACC': 0.26875}